In [230]:
import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# %matplotlib inline

In [231]:
df=pd.read_csv("insurance_claims.csv")
df.columns

Index(['months_as_customer', 'age', 'policy_number', 'policy_bind_date',
       'policy_state', 'policy_csl', 'policy_deductable',
       'policy_annual_premium', 'umbrella_limit', 'insured_zip', 'insured_sex',
       'insured_education_level', 'insured_occupation', 'insured_hobbies',
       'insured_relationship', 'capital-gains', 'capital-loss',
       'incident_date', 'incident_type', 'collision_type', 'incident_severity',
       'authorities_contacted', 'incident_state', 'incident_city',
       'incident_location', 'incident_hour_of_the_day',
       'number_of_vehicles_involved', 'property_damage', 'bodily_injuries',
       'witnesses', 'police_report_available', 'total_claim_amount',
       'injury_claim', 'property_claim', 'vehicle_claim', 'auto_make',
       'auto_model', 'auto_year', 'fraud_reported', '_c39'],
      dtype='object')

'months_as_customer', 'policy_number', 'policy_bind_date', 'umbrella_limit', 'incident_date', 'total_claim_amount', 'auto_model', 'c_39'

# Data cleaning, datatype, index 

In [232]:
# drop column as 100% nan-values
# drop umbrella limits as 80% of values are zeros, std = 1.1 Mio
df.drop(['months_as_customer', 'policy_number',  'umbrella_limit',  'total_claim_amount', '_c39'],axis=1,inplace= True)

In [233]:
df.columns

Index(['age', 'policy_bind_date', 'policy_state', 'policy_csl',
       'policy_deductable', 'policy_annual_premium', 'insured_zip',
       'insured_sex', 'insured_education_level', 'insured_occupation',
       'insured_hobbies', 'insured_relationship', 'capital-gains',
       'capital-loss', 'incident_date', 'incident_type', 'collision_type',
       'incident_severity', 'authorities_contacted', 'incident_state',
       'incident_city', 'incident_location', 'incident_hour_of_the_day',
       'number_of_vehicles_involved', 'property_damage', 'bodily_injuries',
       'witnesses', 'police_report_available', 'injury_claim',
       'property_claim', 'vehicle_claim', 'auto_make', 'auto_model',
       'auto_year', 'fraud_reported'],
      dtype='object')

In [234]:
#check duplicates
df.duplicated().sum() 

np.int64(0)

In [235]:
# df=df.set_index('policy_number')

In [236]:
#ensure consistency on column names
rename ={'capital-gains':'capital_gains','capital-loss': 'capital_loss'}
df.rename(columns=rename, inplace=True)

In [237]:
#change to datetime
df["policy_bind_date"]=pd.to_datetime(df['policy_bind_date'])
df["incident_date"]=pd.to_datetime(df['incident_date'])

In [238]:
#Rename Strings
df["collision_type"]=df["collision_type"].apply(lambda x: "No Collision" if str(x)=="?" else x)
df["police_report_available"]=df["police_report_available"].apply(lambda x: "Unknown" if str(x)=="?" else x)
df["property_damage"]=df["property_damage"].apply(lambda x: "Unknown" if str(x)=="?" else x)

# Set new features

In [239]:
df['insured_zip_class']=df['insured_zip'].astype(str).str[:2]
df.drop('insured_zip', axis=1, inplace=True)

In [240]:
# #additional columns year, month, date, week number of policy bind date
# df['policy_bind_year']=df["policy_bind_date"].dt.year #to be reviewed to fraud_reported
# df['policy_bind_month']=df["policy_bind_date"].dt.month #to be reviewed to fraud_reported
# df['policy_bind_weekday']=df["policy_bind_date"].dt.weekday+1 #to be reviewed to fraud_reported 
# df['policy_bind_week']=df["policy_bind_date"].dt.isocalendar().week #to be reviewed to fraud_reported 

In [241]:
# #additional columns month, date of incident date as year is 2015 for all cases
# df['incident_month']=df["incident_date"].dt.month #to be reviewed to fraud_reported
# df['incident_weekday']=df["incident_date"].dt.weekday+1 #to be reviewed to fraud_reported
# df['incident_week']=df["incident_date"].dt.isocalendar().week

In [242]:
#subtract incident_date from policy_bind date to get policy duration
df['policy_duration_days']=df['incident_date']-df['policy_bind_date']
df['policy_duration_days']=df['policy_duration_days'].dt.days
#df['policy_duration_days'].head() #in days

# # policiy_duration_days to months
# df['policy_duration_months']=df['policy_duration_days']//30

# #age of car at incident date
# df['age_of_auto']=df['incident_date'].dt.year-df['auto_year']

In [243]:
df.drop(['policy_bind_date','incident_date'],axis=1,inplace= True)

In [244]:
#df.to_csv('insurance_claims_clean.csv')

In [245]:
df.head()

,age,policy_state,policy_csl,policy_deductable,policy_annual_premium,insured_sex,insured_education_level,insured_occupation,insured_hobbies,insured_relationship,...,police_report_available,injury_claim,property_claim,vehicle_claim,auto_make,auto_model,auto_year,fraud_reported,insured_zip_class,policy_duration_days
0,48,OH,250/500,1000,1406.91,MALE,MD,craft-repair,sleeping,husband,...,YES,6510,13020,52080,Saab,92x,2004,Y,46,100
1,42,IN,250/500,2000,1197.22,MALE,MD,machine-op-inspct,reading,other-relative,...,Unknown,780,780,3510,Mercedes,E400,2007,Y,46,3130
2,29,OH,100/300,2000,1413.14,FEMALE,PhD,sales,board-games,own-child,...,NO,7700,3850,23100,Dodge,RAM,2007,N,43,5282
3,41,IL,250/500,2000,1415.74,FEMALE,PhD,armed-forces,board-games,unmarried,...,NO,6340,6340,50720,Chevrolet,Tahoe,2014,Y,60,8996
4,44,IL,500/1000,1000,1583.91,MALE,Associate,sales,board-games,unmarried,...,NO,1300,650,4550,Accura,RSX,2009,N,61,256


# Start of One Hote Encodings (only nominal variables)

### Drop columns with p>5%

In [246]:
#remove policy_state
df = df.drop('policy_state', axis=1)
#remove policy_csl
df = df.drop('policy_csl', axis=1)
#remove insured_sex
df=df.drop('insured_sex', axis=1)
#remove insured_education_level
df=df.drop('insured_education_level', axis=1)
#remove insured_occupation
df=df.drop('insured_occupation', axis=1)
#remove insured_relationship
df=df.drop('insured_relationship', axis=1)
#remove incident_city
df=df.drop('incident_city', axis=1)
#remove incident_location
df=df.drop('incident_location', axis=1)
#remove police_report_available
df=df.drop('police_report_available', axis=1)
#remove auto_make
df=df.drop('auto_make', axis=1)
#remove auto_model
df=df.drop('auto_model', axis=1)

#drop insured_zip_class and incident_hour_of_the_day --> I checked p-values not significant
df=df.drop('insured_zip_class', axis=1)
df=df.drop('incident_hour_of_the_day', axis=1)
df.head()

,age,policy_deductable,policy_annual_premium,insured_hobbies,capital_gains,capital_loss,incident_type,collision_type,incident_severity,authorities_contacted,...,number_of_vehicles_involved,property_damage,bodily_injuries,witnesses,injury_claim,property_claim,vehicle_claim,auto_year,fraud_reported,policy_duration_days
0,48,1000,1406.91,sleeping,53300,0,Single Vehicle Collision,Side Collision,Major Damage,Police,...,1,YES,1,2,6510,13020,52080,2004,Y,100
1,42,2000,1197.22,reading,0,0,Vehicle Theft,No Collision,Minor Damage,Police,...,1,Unknown,0,0,780,780,3510,2007,Y,3130
2,29,2000,1413.14,board-games,35100,0,Multi-vehicle Collision,Rear Collision,Minor Damage,Police,...,3,NO,2,3,7700,3850,23100,2007,N,5282
3,41,2000,1415.74,board-games,48900,-62400,Single Vehicle Collision,Front Collision,Major Damage,Police,...,1,Unknown,1,2,6340,6340,50720,2014,Y,8996
4,44,1000,1583.91,board-games,66000,-46000,Vehicle Theft,No Collision,Minor Damage,NaN,...,1,NO,0,1,1300,650,4550,2009,N,256


In [247]:
#train_test_split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop('fraud_reported', axis=1), df['fraud_reported'], test_size=0.2, random_state=42)

In [248]:

from sklearn.preprocessing import OneHotEncoder


In [249]:

#encode insured_hobbies, incident_type, collision_type, incident_severity, authorities_contacted, incident_state
enc=OneHotEncoder(handle_unknown='ignore')
X_train_enc=enc.fit_transform(X_train[['insured_hobbies', 'incident_type', 'collision_type', 'incident_severity', 'authorities_contacted', 'incident_state']]).toarray()
#dataframe
X_train_enc=pd.DataFrame(X_train_enc, columns=enc.get_feature_names_out(['insured_hobbies', 'incident_type', 'collision_type', 'incident_severity', 'authorities_contacted', 'incident_state']))

#reset index
X_train_enc.reset_index(drop=True, inplace=True)
X_train.reset_index(drop=True, inplace=True)

#concatenate
X_train=pd.concat([X_train, X_train_enc], axis=1)
#drop columns
X_train.drop(['insured_hobbies', 'incident_type', 'collision_type', 'incident_severity', 'authorities_contacted', 'incident_state'], axis=1, inplace=True)
X_train




,age,policy_deductable,policy_annual_premium,capital_gains,capital_loss,number_of_vehicles_involved,property_damage,bodily_injuries,witnesses,injury_claim,...,authorities_contacted_Other,authorities_contacted_Police,authorities_contacted_nan,incident_state_NC,incident_state_NY,incident_state_OH,incident_state_PA,incident_state_SC,incident_state_VA,incident_state_WV
0,45,2000,1104.50,0,0,1,NO,2,2,14100,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,23,1000,1099.95,0,-71900,1,NO,1,0,6550,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,45,2000,1221.41,46700,-72500,1,NO,2,1,300,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,26,500,1500.04,0,-36500,1,NO,0,2,860,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,43,1000,974.84,52100,0,1,NO,0,1,21330,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,61,1000,1125.37,0,-56400,3,Unknown,0,2,6650,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
796,55,2000,1589.54,55400,0,3,Unknown,2,0,17060,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
797,42,1000,1023.11,0,-45300,3,NO,1,2,10700,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
798,28,500,1075.41,55200,0,1,NO,1,0,7340,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [250]:
# same for test data
X_test_enc=enc.transform(X_test[['insured_hobbies', 'incident_type', 'collision_type', 'incident_severity', 'authorities_contacted', 'incident_state']]).toarray()
X_test_enc=pd.DataFrame(X_test_enc, columns=enc.get_feature_names_out(['insured_hobbies', 'incident_type', 'collision_type', 'incident_severity', 'authorities_contacted', 'incident_state']))
X_test_enc.reset_index(drop=True, inplace=True)
X_test.reset_index(drop=True, inplace=True)
X_test=pd.concat([X_test, X_test_enc], axis=1)
X_test.drop(['insured_hobbies', 'incident_type', 'collision_type', 'incident_severity', 'authorities_contacted', 'incident_state'], axis=1, inplace=True)
X_test

,age,policy_deductable,policy_annual_premium,capital_gains,capital_loss,number_of_vehicles_involved,property_damage,bodily_injuries,witnesses,injury_claim,...,authorities_contacted_Other,authorities_contacted_Police,authorities_contacted_nan,incident_state_NC,incident_state_NY,incident_state_OH,incident_state_PA,incident_state_SC,incident_state_VA,incident_state_WV
0,26,2000,1137.02,31500,0,1,YES,1,3,16020,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,33,1000,1422.78,61600,0,3,Unknown,2,3,5280,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,51,1000,976.37,0,-61000,3,Unknown,1,3,13520,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,57,2000,1373.21,42700,-64900,3,NO,0,0,6280,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,29,1000,1117.17,0,-29900,1,YES,2,0,620,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,40,2000,1190.60,0,-45300,1,Unknown,1,3,7860,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
196,25,500,1259.02,67000,-53600,1,NO,2,2,940,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
197,48,500,1451.54,0,0,4,NO,2,3,7230,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
198,27,1000,1202.75,57900,-90100,3,NO,0,2,6630,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [251]:
# p-values for insured_zip_class and incident_hour_of_the_day
###import scipy.stats as stats
# insured_zip_class
# H0: insured_zip_class has no effect on fraud_reported
# H1: insured_zip_class has an effect on fraud_reported
# chi2 test
###contingency_table=pd.crosstab(X_train['insured_zip_class'], X_train['fraud_reported'])
###chi2, p, dof, expected=stats.chi2_contingency(contingency_table)
###p
# p-value=0.9999>0.05
# H0 is accepted
# insured_zip_class has no effect on fraud_reported


# incident_hour_of_the_day
# H0: incident_hour_of_the_day has no effect on fraud_reported
# H1: incident_hour_of_the_day has an effect on fraud_reported
# chi2 test
###contingency_table=pd.crosstab(X_train['incident_hour_of_the_day'], X_train['fraud_reported'])
###chi2, p, dof, expected=stats.chi2_contingency(contingency_table)
###p
# p-value=0.9999>0.05
# H0 is accepted
# incident_hour_of_the_day has no effect on fraud_reported